# CET-Epi: Causal Emergence Validation

This notebook validates that the learned macro-scale exhibits **Causal Emergence** (higher Effective Information than micro-scale).

In [ ]:
import sys
sys.path.append('../src')

import torch
import numpy as np
import matplotlib.pyplot as plt

from src.models.cet_epi import CET_Epi
from src.data.chickenpox_loader import MultiScaleChickenpoxLoader
from src.evaluation.ei_analyzer import EIAnalyzer
from src.utils.config import load_config
from src.utils.gpu import setup_gpu

# Setup
device = setup_gpu()
config = load_config('chickenpox.yaml', '../configs')

# Load model (update path to your checkpoint)
checkpoint_path = '../checkpoints/chickenpox/best_model.pt'
checkpoint = torch.load(checkpoint_path, map_location=device)

model = CET_Epi(
    n_micro=config.data.micro_nodes,
    n_macro=config.data.macro_nodes,
    in_channels=config.data.features,
    hidden_dim=config.model.hidden_dim
).to(device)

model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f"Loaded model from epoch {checkpoint['epoch']}")

## 1. Effective Information Over Time

In [ ]:
# Load test data
loader = MultiScaleChickenpoxLoader(lags=config.data.get('lags', 4))
_, test_data = loader.get_split(config.data.train_ratio)

# Analyze EI
analyzer = EIAnalyzer(model, device)
ei_results = analyzer.analyze_over_time(test_data)

# Plot
fig = analyzer.plot_emergence_analysis(ei_results, save_path='../logs/figures/emergence_validation.png')

## 2. Emergence Statistics

In [ ]:
emergence_scores = [r['emergence_score'] for r in ei_results]
positive_ratio = sum(1 for e in emergence_scores if e > 0) / len(emergence_scores)

print(f"Average Emergence Score: {np.mean(emergence_scores):.4f}")
print(f"Median Emergence Score: {np.median(emergence_scores):.4f}")
print(f"Positive Emergence Ratio: {positive_ratio:.1%}")
print(f"Max Emergence: {max(emergence_scores):.4f}")
print(f"Min Emergence: {min(emergence_scores):.4f}")

# Histogram
plt.figure(figsize=(10, 6))
plt.hist(emergence_scores, bins=20, alpha=0.7, edgecolor='black')
plt.axvline(x=0, color='red', linestyle='--', label='No Emergence')
plt.axvline(x=np.mean(emergence_scores), color='green', linestyle='-', 
            label=f'Mean: {np.mean(emergence_scores):.3f}')
plt.xlabel('Emergence Score (EI_macro - EI_micro)')
plt.ylabel('Frequency')
plt.title('Distribution of Causal Emergence Scores')
plt.legend()
plt.savefig('../logs/figures/emergence_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

## 3. Learned Macro Partition

In [ ]:
# Visualize learned clustering
partition = analyzer.get_macro_partition()
print(f"Learned partition: {partition}")
print(f"Region counts: {np.bincount(partition)}")

# Plot
fig = analyzer.visualize_partition(save_path='../logs/figures/learned_partition.png')

## 4. Assignment Matrix Analysis

In [ ]:
# Get assignment from a sample
sample = next(iter(test_data))
with torch.no_grad():
    _, _, int_dict = model(sample.x.to(device), sample.edge_index.to(device), 
                          sample.edge_attr.to(device) if sample.edge_attr is not None else None, 
                          return_all=True)
    S = int_dict['S']

# Visualize
from src.evaluation.visualizer import CET_EpiVisualizer
viz = CET_EpiVisualizer()
viz.plot_assignment_matrix(S, save_name='assignment_matrix_validation.png')